# 05 — Deploy App

Writes `app.yaml` from the widget values, then deploys the Databricks App.

The `app/` folder must already be in the same workspace folder as these notebooks.

Re-run safe — every step is idempotent.

In [ ]:
dbutils.widgets.text("app_name",       "my-first-genie-app", "App Name (lowercase, hyphens only)")
dbutils.widgets.text("warehouse_id",   "",                   "SQL Warehouse ID")
dbutils.widgets.text("genie_space_id", "",                   "Genie Space ID")
dbutils.widgets.text("catalog",        "medtech",            "Catalog")
dbutils.widgets.text("schema",         "sales",              "Schema")

import re, os, time, requests
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.apps import AppDeployment, AppDeploymentMode, AppDeploymentState
from databricks.sdk.service.iam import AccessControlRequest, PermissionLevel

app_name       = dbutils.widgets.get("app_name").strip()
warehouse_id   = dbutils.widgets.get("warehouse_id").strip()
genie_space_id = dbutils.widgets.get("genie_space_id").strip()
catalog        = dbutils.widgets.get("catalog").strip()
schema         = dbutils.widgets.get("schema").strip()

if not warehouse_id:
    raise ValueError("warehouse_id is required")
if not genie_space_id:
    raise ValueError("genie_space_id is required")
if not re.match(r'^[a-z][a-z0-9-]*$', app_name):
    raise ValueError("app_name must be lowercase letters, numbers, and hyphens only")

notebook_ws_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebook_dir     = os.path.dirname(notebook_ws_path)
app_ws_path      = f"{notebook_dir}/../app"
app_fs_path      = f"/Workspace{app_ws_path}"

print(f"App name:    {app_name}")
print(f"Warehouse:   {warehouse_id}")
print(f"Genie space: {genie_space_id}")
print(f"Catalog:     {catalog}")
print(f"Schema:      {schema}")
print(f"App path:    {app_fs_path}")

## Step 1 — Get App Service Principal and Grant Genie Space Access

In [ ]:
w = WorkspaceClient()

app_info     = w.apps.get(name=app_name)
sp_client_id = app_info.service_principal_client_id

if not sp_client_id:
    raise ValueError(f"App '{app_name}' has no service principal yet. Ensure it was created via the UI prereq.")

print(f"App SP client ID: {sp_client_id}")

cfg     = w.config
host    = cfg.host.rstrip("/")
headers = {**dict(cfg.authenticate()), "Content-Type": "application/json"}
acl_body = {"access_control_list": [{"service_principal_name": str(sp_client_id), "permission_level": "CAN_MANAGE"}]}

granted = False
for obj_type in ("genie", "datarooms"):
    for method, fn in (("PATCH", requests.patch), ("PUT", requests.put)):
        url  = f"{host}/api/2.0/permissions/{obj_type}/{genie_space_id}"
        resp = fn(url, headers=headers, json=acl_body)
        print(f"  {method} /permissions/{obj_type}/... \u2192 {resp.status_code}")
        if resp.ok:
            print(f"  Granted CAN_MANAGE via {method} permissions/{obj_type}")
            granted = True
            break
    if granted:
        break

if not granted:
    print()
    print("Automated grant failed. Run this command in a terminal to grant manually:")
    print(f"  databricks permissions update genie {genie_space_id} \\")
    print(f"    --json '{{\"access_control_list\": [{{\"service_principal_name\": \"{sp_client_id}\", \"permission_level\": \"CAN_MANAGE\"}}]}}'")
    print()
    print("Or share the Genie space manually via the UI:")
    print("  Genie > your space > Share > add service principal by client ID > CAN_MANAGE")

## Step 2 — Write app.yaml

In [ ]:
app_yaml = f"""command:
  - uvicorn
  - app:app
  - --host
  - \"0.0.0.0\"
  - --port
  - \"8000\"

auth:
  - user

env:
  - name: DATABRICKS_WAREHOUSE_ID
    value: \"{warehouse_id}\"
  - name: GENIE_SPACE_ID
    value: \"{genie_space_id}\"
  - name: CATALOG
    value: \"{catalog}\"
  - name: SCHEMA
    value: \"{schema}\"

resources:
  - name: sql_warehouse
    sql_warehouse:
      id: {warehouse_id}
      permission: CAN_USE
  - name: genie_space
    genie_space:
      id: {genie_space_id}
      permission: CAN_MANAGE
"""

yaml_path = f"{app_fs_path}/app.yaml"
dbutils.fs.put(yaml_path, app_yaml, overwrite=True)

print(f"app.yaml written to {yaml_path}")
print(app_yaml)

## Step 3 — Deploy App

In [ ]:
print(f"Deploying from {app_ws_path} ...")
deployment = w.apps.deploy(
    app_name=app_name,
    app_deployment=AppDeployment(
        source_code_path=os.path.normpath(app_fs_path),
        mode=AppDeploymentMode.SNAPSHOT
    )
).result()

print(f"Deployment state: {deployment.status.state}")
if deployment.status.state != AppDeploymentState.SUCCEEDED:
    raise Exception(f"Deployment failed: {deployment.status.message}")
print("Deployment succeeded. App should be RUNNING shortly.")

## Step 4 — Grant Unity Catalog Permissions

In [ ]:
for sql in [
    f"GRANT USE_CATALOG ON CATALOG {catalog} TO `{sp_client_id}`",
    f"GRANT USE_SCHEMA ON SCHEMA {catalog}.{schema} TO `{sp_client_id}`",
    f"GRANT SELECT ON SCHEMA {catalog}.{schema} TO `{sp_client_id}`",
]:
    try:
        spark.sql(sql)
        print(f"  OK  {sql}")
    except Exception as e:
        print(f"  ~   {sql}  ({e})")

## Step 5 — Print URL

In [ ]:
print("Waiting for RUNNING state ...")

app_url = None
for i in range(30):
    time.sleep(10)
    data    = requests.get(f"{host}/api/2.0/apps/{app_name}", headers=headers).json()
    state   = (data.get("app_status") or data.get("status") or {}).get("state", "")
    message = (data.get("app_status") or data.get("status") or {}).get("message", "")
    app_url = data.get("url", "")
    print(f"  [{i+1:02d}/30] {state:<12} {message[:60]}")
    if state == "RUNNING":
        break
    if state in ("CRASHED", "STOPPED"):
        print(f"App {state}. Check logs: Apps > {app_name} > Logs")
        break

if app_url:
    displayHTML(f"""
    <div style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;
                padding:28px 32px;background:#fff;border:1px solid #e0e0e0;
                border-radius:12px;max-width:600px;margin:16px 0">
      <div style="color:#EB1700;font-size:22px;font-weight:700;margin-bottom:8px">Setup Complete</div>
      <div style="margin-bottom:16px">
        <span style="font-size:12px;color:#888;text-transform:uppercase;letter-spacing:.5px">App URL</span><br>
        <a href="{app_url}" target="_blank" style="font-size:16px;color:#1565C0">{app_url}</a>
      </div>
      <div>
        <span style="font-size:12px;color:#888;text-transform:uppercase;letter-spacing:.5px">Genie Space ID</span><br>
        <code style="font-size:14px;color:#333">{genie_space_id}</code>
      </div>
    </div>
    """)